# Incremental Embedding Updates at Scale (Part 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qdrant/examples/blob/master/temporal-data-drift/sync_at_scale_with_digests.ipynb)

[Part 1](https://qdrant.tech/documentation/tutorials-operations/incremental-embedding-updates/) keeps a Qdrant collection in sync with changing documentation by reconciling the full list of current chunks against the whole Qdrant collection on every run. It is a simple setup that works for many mid-sized corpora. However, it doesn't scale that well, since every run reads the whole collection, the sync cost grows linearly with the size of the corpus.

This notebook proposes an alternative, more involved approach for the case where scale is a real factor (millions or billions of chunks). On top of the Part 1 idea it adds a small summary layer, so each run scans this summary and touches only the parts of the collection that changed.

## The Idea

The method splits the chunks into a fixed number of **buckets** and gives each bucket a **digest**: a single number that summarizes everything in it, the whole bucket's state.  
A bucket's digest is built from each chunk's digest. A chunk's digest depends on the current chunk's position in the docs (f.e. section of a page) the and its current text. So if any chunk's text or position in the docs changes, a new chunk lands in the bucket, or one leaves it, the bucket's digest changes and we know that bucket needs an inspection. If the digest is unchanged, the bucket's contents are unchanged and we skip it, saving sync time.

We keep two collections:

- the **chunks collection**, whose embeddings serve vector search: the Part 1 collection plus one extra payload field, `sync_bucket`. The point ID is unchanged from Part 1 (a stable ID derived from the chunk's address). We do not store the per-chunk digests; they are cheap to recompute, and only the per-bucket digest is kept.
- a small **summary collection**, holding each bucket and its digest. This is the sync state.

This approach is built on two main ideas:

- **Comparison instead of full scanning.** We compute the digests from the current source (e.g. current state of documentation, for example) and compare them against the digests Qdrant already stored. Only the chunks in the buckets whose digest differs get reconciled; the rest of the chunks collection is never read.
- **Using XOR to combine chunk digests into a bucket digest.**:
  1. it is order-independent, so however the chunks come back, the same set of chunks' digests gives the same bucket digest;
  2. it folds any number of chunk digests into one fixed-width number with one cheap operation, so the summary stays small no matter how big the bucket;
  3. it is reversible (XORing a value twice cancels it), so at scale a bucket's digest could be patched for a single added or removed chunk instead of rebuilt. This notebook recomputes each changed bucket from scratch for clarity, but that reversibility is why XOR is the natural choice.

### The Math Behind The Idea

Say we use 2^16 = 65536 buckets. For a corpus of 1,000,000 chunks that is about 15 chunks per bucket. Each run compares the 65536 digests to find the changed buckets, then reads chunks only from those buckets. If 50 chunks changed, they sit in at most ~50 buckets, so we read on the order of 50 x 15 = 750 chunks, not 1,000,000. Part 1 reads all 1,000,000 every run.

We also could group the 65536 digests into a handful of points rather than storing one bucket in one point of the **summary collection**, to optimize even further. The build section below explains why and how.

For explainability, this notebook uses only **16 buckets**. All the code is written against one constant, `N_BUCKETS`; set it to something reasonable for production use cases.

## Prerequisites

This notebook uses Qdrant Cloud and its Free Tier Inference. Create a Free Tier Qdrant Cloud cluster and paste its URL and API key into the client cell below.

In [ ]:
%pip install -q "qdrant-client>=1.18"

In [2]:
from qdrant_client import QdrantClient, models

# Replace url and api_key with your own from https://cloud.qdrant.io
client = QdrantClient(
    url="https://xyz-example.qdrant.io:6333",
    api_key="<your-api-key>",
    cloud_inference=True
)

## The Chunks

We use the same documentation chunking approach as described in Part 1. Each chunk has an address (which page (url), which section (anchor), which piece of that section (chunk_num)) and its text. Two derived values carry the method:

- `point_id`: a stable ID computed from the address, the chunk's `url`, section `anchor`, and `chunk_num`. Same address, same ID.
- `content_hash`: a fingerprint of the text. Same text, same hash.

In [3]:
CHUNKS = [
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "prerequisites", "chunk_num": 0,
     "text": "Prerequisites - Docker and Docker Compose installed - curl available in your terminal ..."},
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-2-enable-tls", "chunk_num": 0,
     "text": "Step 2: Enable TLS. Generate a local self-signed certificate and point Qdrant at it ..."},
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-3-enable-an-admin-api-key", "chunk_num": 0,
     "text": "Step 3: Enable an Admin API Key. Without authentication, anyone with network access ..."},
]

In [4]:
import hashlib
import uuid


def content_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()


def point_id(url, anchor, num):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{url}#{anchor}::{num}"))


def prepare(chunks):
    prepared = []
    for c in chunks:
        # section_url is the human-readable address; point_id and content_hash are derived once here.
        # (Part 1 also normalizes c["text"] first; left out here to stay on topic.)
        section_url = f'{c["url"]}#{c["anchor"]}' if c["anchor"] else c["url"]
        prepared.append({
            **c,
            "section_url": section_url,
            "content_hash": content_hash(c["text"]),
            "point_id": point_id(c["url"], c["anchor"], c["chunk_num"]),
        })
    return prepared

## Buckets: Where Each Chunk Lives

A bucket is one of `N_BUCKETS` slots (16 is this notebook, something reasonably big in production).  
A chunk's bucket comes from its `point_id`, so editing the text never moves a chunk to a different bucket:

```
point_id = ID from (url, anchor, chunk_num)
bucket   = sha256(point_id) mod N_BUCKETS

example (the "Step 2: Enable TLS" chunk):
  point_id = "4e72de03-624c-5b8e-a3ef-93e3a2d3267b"
  bucket   = sha256(point_id) mod 16  =  11
```

The next cell prints the bucket each of our chunks lands in.

In [5]:
N_BUCKETS = 16
GROUP_SIZE = 4  # buckets packed per group; 16 / 4 = 4 groups


def bucket(pid):
    return int(hashlib.sha256(pid.encode()).hexdigest(), 16) % N_BUCKETS


for c in prepare(CHUNKS):
    print(bucket(c["point_id"]), c["point_id"], c["section_url"])

1 2ff5204a-0353-5991-ba55-acd1995063e8 https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#prerequisites
11 4e72de03-624c-5b8e-a3ef-93e3a2d3267b https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#step-2-enable-tls
12 3ecb6382-a741-5704-9797-b2b8d70847c1 https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/#step-3-enable-an-admin-api-key


## Digests: One Number per Bucket

Each chunk contributes to a bucket one number, chunk's digest, computed from chunk's ID and its content hash (basically, hash of 2 concatenated hashes). If chunk's position or content stay the same, chunk's digest stays the same. A bucket's digest is the XOR of the chunk digests in it:

```
bucket 5 holds two chunks, each contributes a digest:
  chunk A  ->  0011
  chunk B  ->  0101
  digest   =   0011 XOR 0101  =  0110

edit chunk B's text, so its chunk digest changes:
  chunk B  ->  1001
  digest   =   0011 XOR 1001  =  1010      (changed: bucket 5 gets investigated)
```

Because the chunk digest folds in both the ID and the text, any edit, insert, or delete in a bucket changes its digest (a collision is about 2^-60, negligible), and an untouched bucket keeps the exact same digest. That is the signal we compare on.

In [6]:
def chunk_digest(pid, chash):
    # First 15 hex digits of the combined hash = a 60-bit number.
    # 60 bits fits Qdrant's signed 64-bit integer payload, so digests can be stored as plain integers.
    combined = hashlib.sha256((pid + chash).encode()).hexdigest()
    return int(combined[:15], 16)


def compute_digests(chunks):
    digests = [0] * N_BUCKETS
    for c in chunks:
        b = bucket(c["point_id"])
        digests[b] ^= chunk_digest(c["point_id"], c["content_hash"])
    return digests


compute_digests(prepare(CHUNKS))

[0,
 1065001772501011583,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 361274710181928245,
 526478318249294445,
 0,
 0,
 0]

## Storing the Chunks

We store the chunks in a Qdrant collection as in Part 1, with one addition: each point carries its `sync_bucket` in the payload, and we index that field so we can later read a single bucket without scanning the collection.

In [7]:
MAIN = "docs-sync-scale"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if not client.collection_exists(MAIN):
    client.create_collection(
        MAIN,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
    )
    client.create_payload_index(MAIN, "sync_bucket", models.PayloadSchemaType.INTEGER)


def payload(c):
    return {
        "url": c["url"],
        "anchor": c["anchor"],
        "chunk_num": c["chunk_num"],
        "section_url": c["section_url"],
        "text": c["text"],
        "content_hash": c["content_hash"],
        "sync_bucket": bucket(c["point_id"]),
    }


def as_points(chunks):
    points = []
    for c in chunks:
        points.append(models.PointStruct(
            id=c["point_id"],
            vector=models.Document(text=c["text"], model=MODEL),  # embedded by Qdrant Cloud Inference
            payload=payload(c),
        ))
    return points


client.upsert(MAIN, points=as_points(prepare(CHUNKS)), wait=True)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

## The Summary Collection of Digests

We keep the digests in Qdrant, in a small separate collection. We could store one point per bucket, but at true scale that could be hundreds thousands of points (f.e. ~65k for 2^16 buckets), and reading the whole summary would then again mean fetching many points per run.

So we could also group bucket digests into each point, balancing number of reads and updates. For example, for our running 2^16 example, we could have 256 points holding 256 digests each in an array (256 x 256 = 65536). A bucket's digest sits at `point = bucket // 256`, `slot = bucket % 256`.

This notebook uses 16 buckets, so for 4 groups it'll be:

```
  point 0  =  digests for buckets 0, 1, 2, 3
  point 1  =  digests for buckets 4, 5, 6, 7
  point 2  =  digests for buckets 8, 9, 10, 11
  point 3  =  digests for buckets 12, 13, 14, 15

a bucket's digest sits at   point = bucket // 4,   slot = bucket % 4
```

Each digest is a 60-bit number (see the previous cell), which fits Qdrant's signed 64-bit integer payload, so we store digests as plain integers. The vectors can be dummy 1-dimensional values, as this collection is never used for vector search, only for lookups.

In [8]:
META = "docs-sync-digests"
N_META = N_BUCKETS // GROUP_SIZE

if not client.collection_exists(META):
    client.create_collection(
        META,
        vectors_config=models.VectorParams(size=1, distance=models.Distance.COSINE),
    )


def write_meta(digests, groups=None):
    """Store bucket digests in the summary collection, one point per group.

    digests: the full list of N_BUCKETS digests.
    groups:  which group points to (re)write; defaults to all of them.
    """
    if groups is None:
        groups = range(N_META)

    points = []
    for g in groups:
        # group g holds the digests of buckets [g * GROUP_SIZE .. g * GROUP_SIZE + GROUP_SIZE - 1]
        start = g * GROUP_SIZE
        group_digests = digests[start:start + GROUP_SIZE]
        points.append(models.PointStruct(
            id=g,
            vector=[1.0],  # dummy: this collection is never searched
            payload={"group": g, "digests": group_digests},
        ))
    client.upsert(META, points=points, wait=True)


def read_meta():
    """Read the summary back as a flat list of N_BUCKETS digests.

    Retrieves the N_META group points and unpacks each group's digest array
    back into its bucket positions.
    """
    digests = [0] * N_BUCKETS
    for point in client.retrieve(META, ids=list(range(N_META)), with_payload=True):
        g = point.payload["group"]
        group_digests = point.payload["digests"]
        for slot, digest in enumerate(group_digests):
            digests[g * GROUP_SIZE + slot] = digest
    return digests


write_meta(compute_digests(prepare(CHUNKS)))
read_meta()

[0,
 1065001772501011583,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 361274710181928245,
 526478318249294445,
 0,
 0,
 0]

## Syncing

A sync run reconciles the collection with the current source in five steps. We follow the running example: since the last sync, `step-2` was edited, `step-3` was removed, and a new `step-4` was added.

1. Compute the digest summary from the current source. `source = [d0, ..., d11, d12, ...]`
2. Read the stored summary from the summary collection. `stored = [d0', ..., d11', d12', ...]`
3. Take the buckets where the two differ. `changed = [0, 11, 12]`
4. Reconcile each changed bucket against the source, comparing by content hash (see below).
5. Rewrite the summary for the changed groups only, after the data writes.

Reconciling one bucket is a set-diff between the source and what Qdrant stores:

```
bucket 11:   step-2 new_hash   vs   step-2 old_hash    ->  changed  ->  re-embed
bucket 0:    step-4 hash       vs   (absent)           ->  new      ->  embed + insert
bucket 12:   (absent)          vs   step-3 hash        ->  gone     ->  delete
```

Step 5 runs after the writes on purpose: if a run stops halfway, the summary still points at the unfinished bucket, so the next run redoes it. Redoing is harmless because the writes are keyed by ID and content hash.

We build this up one function at a time below, running each so you can see what it returns.

### The Edited Source

Here is the source after a month of edits: `prerequisites` is unchanged, `step-2` is edited, `step-3` is gone, and a new `step-4` appears.

In [9]:
LATEST = [
    # unchanged
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "prerequisites", "chunk_num": 0,
     "text": "Prerequisites - Docker and Docker Compose installed - curl available in your terminal ..."},
    # edited text
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-2-enable-tls", "chunk_num": 0,
     "text": "Step 2: Enable TLS. Generate a certificate with mkcert and set the TLS config keys ..."},
    # step-3 removed; new step-4 added
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-4-restrict-access", "chunk_num": 0,
     "text": "Step 4: Restrict access with read-only API keys for untrusted clients ..."},
]

### Steps 1 to 3: Which Buckets Changed

Compute the digest summary from the edited source, read the stored summary, and take the buckets where they differ. This reads only the small summary, never the chunks collection.

In [10]:
latest = prepare(LATEST)
source = compute_digests(latest)   # digests of the edited source
stored = read_meta()               # digests Qdrant currently holds

changed_buckets = []
for b in range(N_BUCKETS):
    if source[b] != stored[b]:
        changed_buckets.append(b)

changed_buckets

[0, 11, 12]

### `read_bucket`: What a Bucket Holds in Qdrant

**Purpose:** read the chunks currently stored in one bucket.
**Input:** a bucket number.
**Output:** a dict mapping each stored chunk's `point_id` to its `content_hash`.

It filters on the indexed `sync_bucket` field, so only that one bucket is read, not the whole collection.

In [11]:
def read_bucket(b):
    """Return {point_id: content_hash} for every chunk stored in bucket b.

    Pages through the results so nothing is missed in a large bucket.
    """
    stored = {}
    offset = None
    while True:
        points, offset = client.scroll(
            MAIN,
            scroll_filter=models.Filter(
                must=[models.FieldCondition(
                    key="sync_bucket",
                    match=models.MatchValue(value=b),
                )],
            ),
            with_payload=["content_hash"],
            with_vectors=False,
            limit=1000,
            offset=offset,
        )
        for point in points:
            stored[str(point.id)] = point.payload["content_hash"]
        if offset is None:
            return stored

In [12]:
# bucket 11 holds the "step-2" chunk; here is what Qdrant stored for it
read_bucket(11)

{'4e72de03-624c-5b8e-a3ef-93e3a2d3267b': '5f10768bbae2b0a5b3c6bc53947fd9e920411897aba9b3280e12ab704fdc65fb'}

### `reconcile_bucket`: Make One Bucket Match the Source

**Purpose:** bring a single changed bucket in line with the source.
**Input:** the bucket number, and the source chunks in that bucket as `{point_id: chunk}`.
**Output:** the counts `(added, re_embedded, deleted)`.

It set-diffs the source against what is stored: new or content-changed chunks are embedded and upserted, and chunks the source no longer has are deleted.

In [13]:
def reconcile_bucket(b, source_chunks):
    """Make bucket b in Qdrant match source_chunks. Returns (added, re_embedded, deleted)."""
    stored = read_bucket(b)   # {point_id: content_hash} currently in Qdrant

    to_write = []             # new or content-changed chunks: embed and upsert
    added = 0
    re_embedded = 0
    for pid, chunk in source_chunks.items():
        if pid not in stored:
            to_write.append(chunk)        # new chunk in this bucket
            added += 1
        elif stored[pid] != chunk["content_hash"]:
            to_write.append(chunk)        # same chunk, changed text
            re_embedded += 1

    to_delete = []            # chunks Qdrant has but the source no longer does
    for pid in stored:
        if pid not in source_chunks:
            to_delete.append(pid)

    if to_write:
        client.upsert(MAIN, points=as_points(to_write), wait=True)
    if to_delete:
        client.delete(MAIN, points_selector=models.PointIdsList(points=to_delete), wait=True)

    return added, re_embedded, len(to_delete)

### `sync`: The Whole Run

**Purpose:** reconcile the collection with the full current source.
**Input:** the current chunk list.
**Output:** a report of which buckets changed and how many chunks were added, re-embedded, and deleted.

It groups the source by bucket once, finds the changed buckets by comparing summaries (steps 1 to 3), reconciles each one (step 4), then rewrites only the changed groups of the summary, last (step 5).

In [14]:
def sync(latest_chunks):
    latest = prepare(latest_chunks)

    # group the source chunks by bucket once, so each bucket's chunks are ready to hand
    source_by_bucket = {}
    for c in latest:
        b = bucket(c["point_id"])
        source_by_bucket.setdefault(b, {})[c["point_id"]] = c

    # steps 1-3: which buckets changed
    source = compute_digests(latest)
    stored = read_meta()
    changed_buckets = []
    for b in range(N_BUCKETS):
        if source[b] != stored[b]:
            changed_buckets.append(b)

    # step 4: reconcile each changed bucket
    report = {"changed_buckets": changed_buckets, "added": 0, "re_embedded": 0, "deleted": 0}
    for b in changed_buckets:
        source_chunks = source_by_bucket.get(b, {})
        added, re_embedded, deleted = reconcile_bucket(b, source_chunks)
        report["added"] += added
        report["re_embedded"] += re_embedded
        report["deleted"] += deleted

    # step 5: rewrite only the changed groups of the summary, after the data writes
    changed_groups = set()
    for b in changed_buckets:
        changed_groups.add(b // GROUP_SIZE)
    write_meta(source, changed_groups)

    return report

In [15]:
sync(LATEST)

{'changed_buckets': [0, 11, 12], 'added': 1, 're_embedded': 1, 'deleted': 1}

The report's `changed_buckets` lists only the buckets holding an edit, insert, or delete, and only the edited and new chunks get re-embedded. The unchanged `prerequisites` chunk sits in another bucket, so it is never fetched.

## Conclusion

The method in one breath: each chunk gets a bucket from its address and a chunk digest from its id and text; one small collection holds the XOR digest of each bucket; every run compares the current digests against the stored ones and reconciles only the buckets that differ. The Qdrant reads and re-embeddings then track how much changed, not the size of the whole corpus.

**Use Part 1** when the corpus is small, up to roughly 100k chunks: fewer moving parts, nothing extra to maintain.

**Use this approach** when the corpus is large and each run's changes are small next to its size (around a million chunks and up): a run reads a small summary plus only the changed buckets, instead of the whole collection. Raise the bucket count as the corpus grows, so each bucket stays small.